# MovieLens Recommendation System

## 1. Importing the Data and EDA

In [ ]:
import pandas as pd

movies_df = pd.read_csv('./ml-latest-small/movies.csv')
movies_df.head()

In [ ]:
ratings_df = pd.read_csv('./ml-latest-small/ratings.csv')
ratings_df.head()

In [ ]:
movies_df.isna().sum(), ratings_df.isna().sum()

In [ ]:
movies_df.duplicated().sum(), ratings_df.duplicated().sum()

In [ ]:
new_ratings_df = ratings_df.drop('timestamp', axis=1)
new_ratings_df.head()

In [ ]:
print('User ID range: ', (new_ratings_df['userId'].min(), new_ratings_df['userId'].max()))
f"Movie ID range: {(new_ratings_df['movieId'].min(), new_ratings_df['movieId'].max())}"

In [ ]:
new_ratings_df['rating'].value_counts().sort_index()

## 2. Model Building

In [ ]:
from surprise import Reader, Dataset

data = Dataset.load_from_df(new_ratings_df, Reader(rating_scale=(0, 5)))

In [ ]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.2, random_state=2026)

In [ ]:
# Importing relevant libraries
from surprise.model_selection import GridSearchCV
from surprise.prediction_algorithms import SVD, KNNWithMeans, KNNBasic, KNNBaseline
from surprise import accuracy
import numpy as np

### a) SVD

In [ ]:
## Perform a gridsearch with SVD
# This cell may take several minutes to run
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [20, 50],
    'lr_all': [0.005, 0.0005],
    'reg_all': [0.02, 0.04]
}
svd_model = GridSearchCV(
    algo_class=SVD,
    param_grid=param_grid,
    cv=3,
    refit=True,
    measures=['rmse']
)
svd_model.fit(trainset)

In [ ]:
# RMSE on test set
predictions = svd_model.test(testset)
svd_accuracy = accuracy.rmse(predictions)
svd_accuracy

### b) KNNBasic

In [ ]:
# Cross validating with KNNBasic
param_grid = {
    'k': [40, 50],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]
    }
}
knn_basic_model = GridSearchCV(
    algo_class=KNNBasic,
    param_grid=param_grid,
    refit=True,
    measures=['rmse'],
    cv=3
)
knn_basic_model.fit(trainset)

In [ ]:
# RMSE on test set
predictions = knn_basic_model.test(testset)
knn_basic_accuracy = accuracy.rmse(predictions)
knn_basic_accuracy

### c) KNNBaseline

In [ ]:
# Cross validating with KNNBaseline
param_grid = {
    'k': [30, 40],
    'min_k': [5, 10],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]
    }
}
knn_baseline_model = GridSearchCV(
    algo_class=KNNBaseline,
    param_grid=param_grid,
    refit=True,
    measures=['rmse'],
    cv=3
)
knn_baseline_model.fit(trainset)

In [ ]:
# RMSE on test set
predictions = knn_baseline_model.test(testset)
knn_baseline_accuracy = accuracy.rmse(predictions)
knn_baseline_accuracy

### d) KNNWithMeans

In [ ]:
# Cross validating with KNNWithMeans
param_grid = {
    'k': [30, 40],
    'min_k': [5, 10],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]
    }
}
knn_with_means_model = GridSearchCV(
    algo_class=KNNWithMeans,
    param_grid=param_grid,
    refit=True,
    measures=['rmse'],
    cv=3
)
knn_with_means_model.fit(trainset)

In [ ]:
# RMSE on test set
predictions = knn_with_means_model.test(testset)
knn_with_means_accuracy = accuracy.rmse(predictions)
knn_with_means_accuracy

### e) Final Evaluation

In [ ]:
svd_accuracy, knn_basic_accuracy, knn_baseline_accuracy, knn_with_means_accuracy

## 3. Making Predictions

In [ ]:
svd_model.predict(1, 3)

In [ ]:
knn_baseline_model.predict(56, 67).est

In [ ]:
svd_model.predict(56, 67).r_ui

In [ ]:
svd_model.best_params

## 4. Recommendation Function

In [ ]:
def recommendations(user_id, ratings_dataframe, movies_dataframe, k=5):
    # Convert df to surprise compatible data
    data = Dataset.load_from_df(ratings_dataframe, Reader(rating_scale=(0, 5)))
    train_data = data.build_full_trainset()

    # Fit SVD model with the best params on the data
    svd_model = SVD(
        n_factors=100,
        n_epochs=50,
        lr_all=0.005,
        reg_all=0.04
    )
    svd_model.fit(train_data)

    # Create predictions for movies the user has not seen
    all_movie_ids = ratings_dataframe['movieId'].unique()
    user_movie_ids = ratings_dataframe['movieId'].loc[ratings_dataframe['userId'] == user_id].unique()

    predictions = [(movie_id, svd_model.predict(user_id, movie_id).est) for movie_id in all_movie_ids if movie_id not in user_movie_ids]
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Return the movie titles
    top_k = predictions[:k]
    top_k_titles = [
        movies_dataframe.loc[movies_dataframe['movieId'] == movie_id, 'title'].values[0]
        for movie_id, _ in top_k
    ]
    return top_k_titles

In [ ]:
top_k = recommendations(1, new_ratings_df, movies_df)

top_k